# Introduction
This notebook is a clone of the run_quantize.py file for mimicking and debugging experiments without using seml and slurm

# 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 39.3936


In [2]:
# Setting up environment
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

import seml
import re
import shutil

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
HUB_PATH = "/nfs/students/daro/.cache/huggingface/hub/"

if not os.path.exists(HUB_PATH):
    os.makedirs(HUB_PATH)
    print(f"Creating huggingface hub path at {HUB_PATH}")
    
print(f"Setting cache path to {CACHE_PATH}")
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
import logging
logger = logging.getLogger("quant_logger")
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability
Initializing src package
Initializing src package

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface
MemTotal: 1007.72 GB
MemFree: 124.22 GB
MemAvailable: 920.71 GB
Free GPU Memory (GB): 39.3936

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB

################################
Authentication with Hugging Face...
################################

Hugging Face token loaded successfully.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.cache/huggingface/token
Login successfu

# 2. SEML Pipeline

# 2.1 Response Generator

In [4]:
exp_id = "09-17-1-test"

# 2.2 Pipeline

# 3. Quantize Models

In [3]:
import logging
from typing import Optional, Dict, Any
from transformers import PreTrainedTokenizer
from torch.utils.data import DataLoader
from src.algorithms.quantization import QUANT_CONFIGS
from src.algorithms.quantization.aqlm_lora import quantize_aqlm_lora
from src.algorithms.quantization.load_quantized import quantize_aqlm
from src.algorithms.quantization.awq import quantize_awq
from src.algorithms.quantization.bnb import quantize_bnb
from src.algorithms.quantization.hqq import quantize_hqq
from src.algorithms.quantization.hqq_plus import quantize_hqq_plus
from src.algorithms.quantization.quanto import quantize_quanto

logger = logging.getLogger("quant_logger")

def quantize(
    model_name: str,
    tokenizer: PreTrainedTokenizer,
    quantize_method: str,
    calib_dataloader: Optional[DataLoader] = None,
    train_dataloader: Optional[DataLoader] = None,
    save_model: bool = False,
    save_path: str = "",
    device: str = "cuda"
) -> Any:
    """
    Quantize a model using the specified method.

    Args:
        model_name (str): Name of the model to quantize.
        tokenizer (PreTrainedTokenizer): Tokenizer for the model.
        quantize_method (str): Quantization method to use.
        calib_dataloader (Optional[DataLoader]): Dataloader for calibration data.
        train_dataloader (Optional[DataLoader]): Dataloader for training data.
        save_model (bool): Whether to save the quantized model.
        save_path (str): Path to save the quantized model.
        device (str): Device to use for quantization.

    Returns:
        Any: Quantized model.

    Raises:
        ValueError: If an invalid quantization method is specified.
    """
    logger.info(f"Quantizing model {model_name} with method {quantize_method}")
    if calib_dataloader:
        logger.info(f"Using calibration data: {calib_dataloader.dataset.__class__.__name__}")

    quantization_functions: Dict[str, callable] = {
        "BNB-4": quantize_bnb,
        "BNB-8": quantize_bnb,
        "AWQ-4": quantize_awq,
        "HQQ-8-uniform": quantize_hqq,
        "HQQ-mixed": quantize_hqq,
        "HQQ-LORA": quantize_hqq_plus,
        "QUANTO": quantize_quanto,
        "QUANTO-CALIB": quantize_quanto,
        "QUANTO-QAT": quantize_quanto,
        "AQLM-PREQUANTIZED": quantize_aqlm,
        "AQLM-LORA": quantize_aqlm_lora
    }

    if quantize_method not in quantization_functions:
        raise ValueError(f"Invalid quantization method. Supported methods are: {', '.join(quantization_functions.keys())}")

    quantize_func = quantization_functions[quantize_method]
    quantize_config = QUANT_CONFIGS[quantize_method]

    try:
        common_args = {
            "model_name": model_name,
            "quantize_config": quantize_config,
            "save_model": save_model,
            "save_path": save_path,
            "device": device
        }

        if quantize_method in ["AWQ-4", "HQQ-LORA", "AQLM-LORA"]:
            common_args["tokenizer"] = tokenizer
            common_args["calib_dataloader"] = calib_dataloader

        if quantize_method in ["QUANTO", "QUANTO-CALIB", "QUANTO-QAT"]:
            common_args["calib_dataloder"] = calib_dataloader
            common_args["train_dataloader"] = train_dataloader

        model = quantize_func(**common_args)
        logger.info(f"Successfully quantized model using {quantize_method}")
        return model

    except Exception as e:
        logger.error(f"Error during quantization with {quantize_method}: {str(e)}")
        raise

ImportError: cannot import name 'DEBUG' from partially initialized module 'src.algorithms.quantization.config' (most likely due to a circular import) (/nfs/homedirs/daro/git/quantization-reliability/src/algorithms/quantization/config/__init__.py)

# 4. Test